# 01 Data: index, duplicate groups, folds, wound crops (local, CPU)
Builds `data/processed/index.csv` and the crop/full images from `data/raw/`.

In [ ]:
import glob, os, sys
if os.path.isdir("/kaggle/input"):
    hits = glob.glob("/kaggle/input/**/src/woundvlm/__init__.py", recursive=True)
    assert hits, "attach the project dataset (lowerlimbwoundvlm) as an input"
    SRC = hits[0].rsplit("/woundvlm/", 1)[0]
else:
    SRC = os.path.abspath("../src")
sys.path.insert(0, SRC)
from woundvlm import env
E = {'project': os.path.abspath('..'), 'in_kaggle': False}
P = E["project"]
print(E)

In [ ]:
from woundvlm import experiments, data
df = experiments.prepare(P)
print(df.groupby(['source', 'status']).size())

In [ ]:
import pandas as pd
lab, azh = data.labelled(df), data.azh_wounds(df)
audit = pd.concat([
    df.groupby(['source', 'status', 'label']).size().rename('n').reset_index().assign(table='status'),
    lab.groupby(['fold', 'label']).size().rename('n').reset_index().assign(table='folds'),
    azh.groupby(['azh_split', 'label']).size().rename('n').reset_index().assign(table='azh')])
os.makedirs(os.path.join(P, 'results'), exist_ok=True)
audit.to_csv(os.path.join(P, 'results', '01_data_audit.csv'), index=False)
assert (lab.groupby('group').fold.nunique() == 1).all()
lab.groupby(['fold', 'label']).size().unstack()